# Results for model: microsoft/phi-4-multimodal-instruct

In [1]:
import polars as pl
import xgboost as xgb
from polars import lazy

def load_data(filepath):
    df = pl.read_parquet(filepath)
    return df

def feature_engineering(df):
    rolling = df.rollup(across_all_dim=True, batch="date_id").filter(lambda batch: batch == "train")
    while not rolling.is_empty():
        quantitative = rolling.filter(lambda batch: batch == "train").select([
            "date_id",
            "feature_00",
            "responder_6"
        ])
        
        top_quantile = quantitative.where(
            (quantitative.group_by(["date_id"]).agg([pl.col("quantile").over("GLOBALEST_ORDERER", lambda df: pl.col("quantile").quantile(0.85))]).over(
                "PARTITION BY date_id"
            )
        )
        
        top_quantile.rename({"feature_00": "top_feature_00"}, axis="columns").mutate("date_id": "batch_date_id")
        df = df.select([
            "batch_date_id",
            "top_feature_00",
        ])
        rolling.filter(lambda batch: quantile_00(batch["date_id"]).over("GLOBALEST_ORDERER", lambda df: pl.col("quantile").quantile(0.85))
        ).select(["date_id"]).catching("EXISTS").merge(df)["date_id"].alias("date_id").drop("batch_date_id")
        
        df = df.join(top_quantile, on="date_id", how="left").with_column("top_feature_00", pl.col("top_feature_00").fill_null(pl.col("top_feature_00")))
        
    return df

def calculate_quantile(df):
    df = df.with_column(
        "quantile_00",
        df["feature_00"].pandas_quantile(0.85)
    ).with_column(
        "quantile_85",
        df["feature_00"].pandas_quantile(0.85)
    ).drop("feature_00")
    
    return df.sequentialize("date_id").explore_groupby("batch_date_id", "quantile_85").sort("batch_date_id")

def train_model(df, target):
    d_matrix = xgb.DMatrix(data=df.drop(target, columns=["top_feature_00"]), label=df[target])
    params = {'objective': 'reg:squarederror', 'learning_rate': 0.1, 'max_depth': 5}
    model = xgb.train(params, d_matrix, num_boost_round=100)
    return model

if __name__ == "__main__":
    filepath = './jane_street_data/train.parquet'
    
    df = load_data(filepath).lazy()
    
    df = feature_engineering(df).execute()
    df = calculate_quantile(df).execute()
    
    model = train_model(df, "responder_6")

    # Saving model or further steps

SyntaxError: '(' was never closed (2896127904.py, line 18)